<a href="https://colab.research.google.com/github/nihemelandu/causal-price-demand/blob/main/notebooks/01_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Setup and Reload
# Purpose: Load raw data and re-establish working environment

from datasets import load_dataset
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Loading FreshRetailNet-50K...")
dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
df = dataset['train'].to_pandas()
df['dt'] = pd.to_datetime(df['dt'])

print(f"Raw dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range: {df['dt'].min().date()} to {df['dt'].max().date()}")
print(f"Store-product pairs: {df.groupby(['store_id','product_id']).ngroups:,}")

In [ ]:
# Cell 2: Stockout Censoring Treatment
# Purpose: Remove fully censored demand observations where stock was zero all day
# Justification: sale_amount = 0 when stock_hour6_22_cnt = 0 reflects supply
# constraint, not true zero demand. Including these would bias elasticity downward.

print("=== STEP 1: STOCKOUT CENSORING TREATMENT ===\n")

n_before = len(df)

# Flag fully censored days
df['fully_stocked_out'] = df['stock_hour6_22_cnt'] == 0

# Report before exclusion
print(f"Records before exclusion:          {n_before:,}")
print(f"Fully stocked-out records:         {df['fully_stocked_out'].sum():,} ({df['fully_stocked_out'].mean()*100:.1f}%)")

# Exclude fully stocked-out days
df_clean = df[~df['fully_stocked_out']].copy()

n_after = len(df_clean)
print(f"Records after exclusion:           {n_after:,}")
print(f"Records removed:                   {n_before - n_after:,} ({(n_before - n_after)/n_before*100:.1f}%)")

# Verify remaining zero-sale records
zero_remaining = (df_clean['sale_amount'] == 0).sum()
print(f"\nRemaining zero-sale records:       {zero_remaining:,} ({zero_remaining/n_after*100:.1f}%)")
print("(These are true zero-demand days — stock was available but nothing sold)")

# Check panel balance after exclusion
obs_per_pair = df_clean.groupby(['store_id','product_id'])['dt'].count()
print(f"\nObservations per store-product pair after exclusion:")
print(f"  Min: {obs_per_pair.min()}")
print(f"  Max: {obs_per_pair.max()}")
print(f"  Mean: {obs_per_pair.mean():.1f}")
print(f"  Median: {obs_per_pair.median():.1f}")
print(f"\nNote: Panel is now UNBALANCED — pairs differ in available days")
print(f"DML will be estimated on the available observations per pair")

In [ ]:
# Cell 3: Anomalous Discount Record Investigation and Exclusion
# Purpose: Handle discount > 1.0 and discount = 0.0 edge cases

print("=== STEP 2: ANOMALOUS DISCOUNT RECORDS ===\n")

# --- Discount > 1.0 ---
print("--- Records with discount > 1.0 ---")
over_one = df_clean[df_clean['discount'] > 1.0]
print(f"Count: {len(over_one)}")
print(over_one[['store_id','product_id','dt','discount','sale_amount',
                'activity_flag']].head(20))
print("\nDecision: EXCLUDE — discount > 1.0 is economically implausible")
print("(No product sells above its reference price in a promotional dataset)")

# --- Discount = 0.0 ---
print("\n--- Records with discount = 0.0 (100% off) ---")
zero_discount = df_clean[df_clean['discount'] == 0.0]
print(f"Count: {len(zero_discount)}")
print("\nSummary of zero-discount records:")
print(zero_discount[['sale_amount','activity_flag','holiday_flag',
                      'stock_hour6_22_cnt']].describe())
print("\nActivity flag distribution in zero-discount records:")
print(zero_discount['activity_flag'].value_counts())
print("\nSale amount distribution in zero-discount records:")
print(zero_discount['sale_amount'].describe())
print("\nDecision: EXCLUDE — discount = 0.0 likely represents free promotional")
print("samples or data entry errors. These cannot be interpreted as a price")
print("signal and would distort log-transformation of the treatment variable.")

# Apply exclusions
n_before_anomaly = len(df_clean)
df_clean = df_clean[
    (df_clean['discount'] <= 1.0) &
    (df_clean['discount'] > 0.0)
].copy()
n_after_anomaly = len(df_clean)

print(f"\nRecords before anomaly exclusion:  {n_before_anomaly:,}")
print(f"Records after anomaly exclusion:   {n_after_anomaly:,}")
print(f"Records removed:                   {n_before_anomaly - n_after_anomaly:,}")

print(f"\nDiscount range after cleaning:")
print(f"  Min: {df_clean['discount'].min():.4f}")
print(f"  Max: {df_clean['discount'].max():.4f}")